In [1]:
# Core imports
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# For pretty printing
import json

print("✅ Imports successful!")
print("📦 Ready to build your first graph!")

✅ Imports successful!
📦 Ready to build your first graph!


In [2]:
class State(TypedDict):
    text:str
def node_a(state:State)->dict:
    return {"text":state["text"]+"a"}
def node_b(state:State)->dict:
    return {"text":state["text"]+"b"}

In [3]:
graph = StateGraph(State)
graph.add_node("node_a",node_a)
graph.add_node("node_b",node_b)

graph.add_edge(START,"node_a")
graph.add_edge("node_a","node_b")   
graph.add_edge("node_b",END)

In [4]:
import base64
from IPython.display import Image, display

app = graph.compile()
mermaid_text = app.get_graph().draw_mermaid()

# Encode the mermaid syntax as base64 and use mermaid.ink as an image source
graph_bytes  = mermaid_text.encode("utf-8")
base64_str   = base64.urlsafe_b64encode(graph_bytes).decode("utf-8")
url          = f"https://mermaid.ink/img/{base64_str}?theme=default"

display(Image(url=url))

In [7]:
initial_state = {"text":""}
final_state = app.invoke(initial_state)
print(json.dumps(final_state, indent=2))

{
  "text": "ab"
}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [14]:
class RecipeState(TypedDict):
    ingredients: list
    steps: list
    formatted_ingredients: str
    formatted_steps: str
    is_valid: bool
    final_recipe: str

In [ ]:
def format_ingredients(state: RecipeState) -> dict:
    formatted = "\n".join([f"• {ing.capitalize()}" for ing in state["ingredients"]])
    return {"formatted_ingredients": formatted}

def format_steps(state: RecipeState) -> dict:
    formatted = "\n".join([f"{i+1}. {step}" for i, step in enumerate(state["steps"])])
    return {"formatted_steps": formatted}

def validate_recipe(state: RecipeState) -> dict:
    is_valid = len(state["ingredients"]) > 0 and len(state["steps"]) > 0
    return {"is_valid": is_valid}

def compile_output(state: RecipeState) -> dict:
    if state["is_valid"]:
        final = f"INGREDIENTS:\n{state['formatted_ingredients']}\n\nSTEPS:\n{state['formatted_steps']}"
    else:
        final = "Invalid recipe: missing ingredients or steps"
    return {"final_recipe": final}

recipe_graph = StateGraph(RecipeState)
recipe_graph.add_node("format_ing", format_ingredients)
recipe_graph.add_node("format_steps", format_steps)
recipe_graph.add_node("validate", validate_recipe)
recipe_graph.add_node("compile", compile_output)

recipe_graph.add_edge(START, "format_ing")
recipe_graph.add_edge("format_ing", "format_steps")
recipe_graph.add_edge("format_steps", "validate")
recipe_graph.add_edge("validate", "compile")
recipe_graph.add_edge("compile", END)

recipe_app = recipe_graph.compile()

In [ ]:
# Test data
test_recipe = {
    "ingredients": ["flour", "eggs", "milk", "butter"],
    "steps": [
        "Mix flour and eggs",
        "Add milk gradually",
        "Melt butter and fold in"
    ],    
}

# Uncomment when ready:
result = recipe_app.invoke(test_recipe)
print(result['final_recipe'])

INGREDIENTS:
• Flour
• Eggs
• Milk
• Butter

STEPS:
1. Mix flour and eggs
2. Add milk gradually
3. Melt butter and fold in


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [17]:
class CalcState(TypedDict):
    value: float      # Current value
    operations: list  # Log of operations performed

def add_10(state: CalcState) -> dict:
    """Add 10 to the current value."""
    new_value = state["value"] + 10
    operations = state["operations"] + [f"Added 10: {state['value']} + 10 = {new_value}"]
    print(f"➕ Adding 10: {state['value']} → {new_value}")
    return {"value": new_value, "operations": operations}

def multiply_by_2(state: CalcState) -> dict:
    """Multiply the current value by 2."""
    new_value = state["value"] * 2
    operations = state["operations"] + [f"Multiplied by 2: {state['value']} × 2 = {new_value}"]
    print(f"✖️  Multiplying by 2: {state['value']} → {new_value}")
    return {"value": new_value, "operations": operations}

def subtract_5(state: CalcState) -> dict:
    """Subtract 5 from the current value."""
    new_value = state["value"] - 5
    operations = state["operations"] + [f"Subtracted 5: {state['value']} - 5 = {new_value}"]
    print(f"➖ Subtracting 5: {state['value']} → {new_value}")
    return {"value": new_value, "operations": operations}

# Build calculator graph
calc_graph = StateGraph(CalcState)

calc_graph.add_node("add", add_10)
calc_graph.add_node("multiply", multiply_by_2)
calc_graph.add_node("subtract", subtract_5)

# Chain: START → add → multiply → subtract → END
calc_graph.add_edge(START, "add")
calc_graph.add_edge("add", "multiply")
calc_graph.add_edge("multiply", "subtract")
calc_graph.add_edge("subtract", END)

calc_app = calc_graph.compile()

print("✅ Calculator ready!")

✅ Calculator ready!


In [19]:
# Test the calculator: (5 + 10) × 2 - 5 = ?
result = calc_app.invoke({
    "value": 5.0,
    "operations": []
})

print("\n" + "="*60)
print("🧮 Calculator Results:")
print("="*60)
print(f"Starting value: 5.0")
print(f"Final value: {result['value']}")
print("\nOperation Log:")
print(result['operations'])
for op in result['operations']:
    print(f"  • {op}")
print("="*60)

➕ Adding 10: 5.0 → 15.0
✖️  Multiplying by 2: 15.0 → 30.0
➖ Subtracting 5: 30.0 → 25.0

🧮 Calculator Results:
Starting value: 5.0
Final value: 25.0

Operation Log:
['Added 10: 5.0 + 10 = 15.0', 'Multiplied by 2: 15.0 × 2 = 30.0', 'Subtracted 5: 30.0 - 5 = 25.0']
  • Added 10: 5.0 + 10 = 15.0
  • Multiplied by 2: 15.0 × 2 = 30.0
  • Subtracted 5: 30.0 - 5 = 25.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
from ast import Or


class EmailState(TypedDict):
    sender: str
    subject: str
    content: str
    classification: str
    action_taken: str

def classify_email(state: EmailState) -> dict:
    """Classify email based on content."""
    if(status is "urgent" Or "asp" for status in state["content"])
    pass

def route_email(state: EmailState) -> Literal["urgent", "spam", "automated", "normal"]:
    """Route email based on classification."""
    # TODO: Return the appropriate route based on classification
    pass

def escalate_to_priority(state: EmailState) -> dict:
    """Handle urgent emails."""
    # TODO: Implement urgent handler
    pass

def delete_email(state: EmailState) -> dict:
    """Handle spam."""
    # TODO: Implement spam handler
    pass

def auto_respond(state: EmailState) -> dict:
    """Handle automated emails."""
    # TODO: Implement automated handler
    pass

def standard_processing(state: EmailState) -> dict:
    """Handle normal emails."""
    # TODO: Implement normal handler
    pass

# TODO: Build the graph
# email_graph = StateGraph(EmailState)
# ... add nodes and edges ...
# email_app = email_graph.compile()

In [9]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check API key
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found! Please:\n"
        "1. Create a .env file in the project root\n"
        "2. Add: OPENAI_API_KEY=your_key_here\n"
        "3. Get a key from https://platform.openai.com/api-keys"
    )

print("✅ Imports successful!")
print("🤖 Ready to build AI agents!")

✅ Imports successful!
🤖 Ready to build AI agents!


In [10]:
from langgraph.graph.message import add_messages as add_msg_fn
existing = [HumanMessage(content="Hello")]
new = [AIMessage(content="Hi there!")]
result = add_msg_fn(existing,new)
print(result)
for msg in result:
    print(f"{msg.content}")

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='54bd9ee4-254a-4fd4-ba68-1e4a63040fc7'), AIMessage(content='Hi there!', additional_kwargs={}, response_metadata={}, id='4f781a8e-a52c-462d-8557-a81edc72adbc', tool_calls=[], invalid_tool_calls=[])]
Hello
Hi there!


In [11]:
@tool
def get_weather(city:str)->str:
    """Get the current weather for the given city"""
    weather_data ={
        "paris":"Sunny,22 C"
    }
    city_lower=city.lower()
    if city_lower in weather_data:
        return f"The Weather in {city_lower} is {weather_data[city_lower]}"
    else:
        return f"Sorry"
print(get_weather.invoke({"city":"parids"}))
print(get_weather)

Sorry
name='get_weather' description='Get the current weather for the given city' args_schema=<class 'langchain_core.utils.pydantic.get_weather'> func=<function get_weather at 0x7b569758ad40>


In [12]:
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)
tools =[get_weather]
llm_with_tools = llm.bind_tools(tools)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
from typing import Literal

# State for our weather agent
class WeatherAgentState(TypedDict):
    messages: Annotated[list, add_messages]
# Node 1: Call the LLM
def call_model(state: WeatherAgentState) -> dict:
    """Call the LLM with the current messages."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Node 2: Execute tools
def call_tools(state: WeatherAgentState) -> dict:
    """Execute the tools requested by the LLM."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # Get tool calls from the last message
    tool_calls = last_message.tool_calls
    
    # Execute each tool
    tool_messages = []
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"🔧 Calling tool: {tool_name}({tool_args})")
        
        # Find and execute the tool
        tool = {t.name: t for t in tools}[tool_name]
        result = tool.invoke(tool_args)
        
        print(f"📊 Tool result: {result}")
        
        # Create a tool message with the result
        tool_messages.append(
            ToolMessage(
                content=result,
                tool_call_id=tool_call["id"]
            )
        )
    
    return {"messages": tool_messages}

# Routing function: Should we continue or end?
def should_continue(state: WeatherAgentState) -> Literal["tools", "__end__"]:
    """Determine if we should call tools or end."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # If the LLM makes tool calls, route to tools
    if last_message.tool_calls:
        print("🔀 LLM requested tool calls, routing to tools")
        return "tools"
    # Otherwise, end
    print("🏁 No more tool calls, ending")
    return "__end__"

In [ ]:
weather_graph =StateGraph(WeatherAgentState)
weather_graph.add_node("agent",call_model)
weather_graph.add_node("tools",call_tools)
weather_graph.add_edge(START,"agent")
weather_graph.add_conditional_edges(
    "agent",
    should_continue,
)




In [14]:
# Example: LLM decides to call multiple tools in parallel
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# Simulating what GPT-4o returns when it wants to call tools in parallel:
parallel_response = AIMessage(
    content="",
    tool_calls=[
        {"name": "get_weather", "args": {"city": "Paris"}, "id": "call_1"},
        {"name": "get_weather", "args": {"city": "London"}, "id": "call_2"},
        {"name": "get_weather", "args": {"city": "Berlin"}, "id": "call_3"},
    ]
)
for tc in parallel_response.tool_calls:
    print(tc['name'])

get_weather
get_weather
get_weather


In [16]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together.
    
    Args:
        a: First number
        b: Second number
        
    Returns:
        The sum of a and b
    """
    result = a + b
    print(f"  ➕ {a} + {b} = {result}")
    return result

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together.
    
    Args:
        a: First number
        b: Second number
        
    Returns:
        The product of a and b
    """
    result = a * b
    print(f"  ✖️  {a} × {b} = {result}")
    return result

@tool
def power(base: float, exponent: float) -> float:
    """Raise a number to a power.
    
    Args:
        base: The base number
        exponent: The exponent
        
    Returns:
        base raised to the power of exponent
    """
    result = base ** exponent
    print(f"  🔢 {base}^{exponent} = {result}")
    return result
math_tools =[add,multiply,power]
math_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)
math_llm_with_tools =math_llm.bind_tools(math_tools)

In [ ]:
from email import message


class MathAgentState(TypedDict):
    message: Annotated[list,add_messages]
def call_math_model(state:MathAgentState)->dict:
    message =state["message"]
    response = math_llm_with_tools.invoke(message)
    return {"message":[response]}

In [9]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
import sqlite3

load_dotenv()

print("✅ Imports successful!")
print("💾 Ready to add memory to agents!")


✅ Imports successful!
💾 Ready to add memory to agents!


In [2]:
# State definition
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

# Create LLM
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.7)

# Node: Call LLM
def chat_node(state: ChatState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# Build graph
builder = StateGraph(ChatState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# KEY DIFFERENCE: Add checkpointer when compiling!
memory = MemorySaver()
chatbot = builder.compile(checkpointer=memory)

print("✅ Chatbot with memory created!")

✅ Chatbot with memory created!


In [10]:
config = {"configurable":{"thread_id":"user-001"}}
result1 = chatbot.invoke(
    {"messages":[HumanMessage(content="My name is rohit and i love python")]},
    config
)
print(result1['messages'][-1].content)
result2 = chatbot.invoke(
    {"messages":[HumanMessage(content="What is my name")]},
     config)
print(result2['messages'][-1].content)


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Hello again, Rohit! It’s great to see your passion for Python. If you have any specific questions, projects, or topics related to Python you'd like to discuss or need help with, please let me know. Whether it's coding challenges, learning resources, or just general discussion about Python's capabilities, I'm here to assist. What are you currently working on or interested in learning more about in Python?


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Your name is Rohit. How can I assist you further today?


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [10]:
db_path ="chatbot_memory.db"
conn = sqlite3.connect(db_path,check_same_thread=False)
persistent_memory = SqliteSaver(conn)
persistent_chatbot = builder.compile(checkpointer=persistent_memory)

NameError: name 'builder' is not defined

In [15]:
from langchain_core.tools import tool
from typing import Literal
from langchain_core.messages import ToolMessage
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
# Enhanced state for support bot
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    issue_status: str  # "open", "in_progress", "resolved"
    issue_type: str    # "technical", "billing", "general"
    customer_name: str

# Tools for support bot
@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for solutions."""
    kb = {
        "password reset": "To reset your password, go to Settings > Security > Reset Password.",
        "billing": "For billing inquiries, check your account dashboard or contact billing@example.com.",
        "technical": "For technical issues, try restarting the application first."
    }
    for key, answer in kb.items():
        if key in query.lower():
            return answer
    return "No specific article found. Escalating to human support."

@tool
def update_issue_status(status: Literal["open", "in_progress", "resolved"]) -> str:
    """Update the status of the customer issue."""
    return f"Issue status updated to: {status}"

# Create support bot
support_tools = [search_knowledge_base, update_issue_status]
support_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)
support_llm_with_tools = support_llm.bind_tools(support_tools)

def support_agent(state: SupportState) -> dict:
    response = support_llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def call_support_tools(state: SupportState) -> dict:
    last_message = state["messages"][-1]
    tool_messages = []
    
    for tool_call in last_message.tool_calls:
        tool = {t.name: t for t in support_tools}[tool_call["name"]]
        result = tool.invoke(tool_call["args"])
        tool_messages.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )
    
    return {"messages": tool_messages}

def support_should_continue(state: SupportState) -> Literal["tools", "__end__"]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return "__end__"

# Build support bot graph
support_builder = StateGraph(SupportState)
support_builder.add_node("agent", support_agent)
support_builder.add_node("tools", call_support_tools)
support_builder.add_edge(START, "agent")
support_builder.add_conditional_edges("agent", support_should_continue)
support_builder.add_edge("tools", "agent")

# Compile with persistent memory
import os
if os.path.exists("support_bot.db"):
    os.remove("support_bot.db")

support_db = sqlite3.connect("support_bot.db", check_same_thread=False)
support_memory = SqliteSaver(support_db)
support_bot = support_builder.compile(checkpointer=support_memory)

print("✅ Customer Support Bot ready!")

✅ Customer Support Bot ready!


In [19]:
# Simulate a support conversation
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import uuid

ticket_config = {"configurable": {"thread_id": f"ticket-{uuid.uuid4()}"}}

# Initial state
initial_state = {
    "messages": [HumanMessage(content="Hi, I forgot my password and can't log in.")],
    "issue_status": "open",
    "issue_type": "technical",
    "customer_name": "Jane Doe"
}

print("=" * 60)
print("Support Ticket #12345")
print("=" * 60)

result1 = support_bot.invoke(initial_state, ticket_config)
print(f"\n🧑 Jane: {initial_state['messages'][0].content}")
print(f"🤖 Support: {result1['messages'][-1].content}")

# Follow-up
result2 = support_bot.invoke(
    {"messages": [HumanMessage(content="Thanks! That worked. My issue is resolved.")]},
    ticket_config
)
print(f"\n🧑 Jane: Thanks! That worked. My issue is resolved.")
print(f"🤖 Support: {result2['messages'][-1]}")

print("\n" + "=" * 60)
print("✅ Conversation and ticket status saved to database!")
print("=" * 60)

Support Ticket #12345


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



🧑 Jane: Hi, I forgot my password and can't log in.
🤖 Support: I've escalated your issue regarding the forgotten password to human support for further assistance. Meanwhile, the status of your issue has been updated to "in progress." If you need any more help or have any other questions, feel free to ask!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



🧑 Jane: Thanks! That worked. My issue is resolved.
🤖 Support: content='Great to hear that your issue is resolved! If you need any more assistance in the future, just let me know. Have a wonderful day!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 270, 'total_tokens': 300, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_5b6a502ecb', 'id': 'chatcmpl-EHC4sz7e02R2Vloz5uNXGkXuHa2uc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a03f33-1d7e-7613-a8f8-eac18b04c6cf-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 270, 'output_tokens': 30, 'total_tokens': 300, 'input_token_details': {'audio':

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [1]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
import sqlite3

load_dotenv()
print("✅ Ready for human-in-the-loop workflows with LangGraph 1.1.9!")

✅ Ready for human-in-the-loop workflows with LangGraph 1.1.9!


In [5]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict

class ApprovalState(TypedDict):
    task: str
    plan: str
    approved: bool
    result: str

def plan_task(state: ApprovalState) -> dict:
    """Generate a plan and ask for approval."""
    plan = f"I will execute: {state['task']} using the following steps: 1. Setup, 2. Execute, 3. Verify"
    print(f"📋 Generated plan: {plan}")
    
    # Request human approval — execution pauses here
    # ⚠️ Note: plan_task() will re-run on resume — idempotency matters!
    approval = interrupt({
        "question": "Do you approve this plan?",
        "plan": plan,
        "options": ["yes", "no", "modify"],
    })
    
    print(f"👤 Human responded: {approval}")
    return {"plan": plan, "approved": approval == "yes"}

def execute_task(state: ApprovalState) -> dict:
    if not state["approved"]:
        return {"result": "❌ Task cancelled by user"}
    result = f"✅ Successfully executed: {state['task']}"
    print(result)
    return {"result": result}

# Build graph
db = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(db)

builder = StateGraph(ApprovalState)
builder.add_node("plan", plan_task)
builder.add_node("execute", execute_task)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)

app = builder.compile(checkpointer=memory)
print("✅ Approval workflow (modern pattern) ready!")

✅ Approval workflow (modern pattern) ready!


In [8]:
config = {"configurable": {"thread_id": "task-001"}}

# --- STEP 1: Run until interrupt ---
print("=== Step 1: Starting task ===")
result = app.invoke(
    {"task": "Deploy database migration", "plan": "", "approved": False, "result": ""},
    config
)
print(f"\n Graph paused. Interrupt data:{result.get('__interrupt__')}")
final_result = app.invoke(Command(resume="no"), config)
print(f"\nFinal result: {final_result['result']}")

=== Step 1: Starting task ===
📋 Generated plan: I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify

 Graph paused. Interrupt data:[Interrupt(value={'question': 'Do you approve this plan?', 'plan': 'I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify', 'options': ['yes', 'no', 'modify']}, id='a1d898868c12ec57ef7a36499277cd93')]
📋 Generated plan: I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify
👤 Human responded: no

Final result: ❌ Task cancelled by user


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [10]:
import sqlite3
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

# ── State ──────────────────────────────────────────────────────────────────
class ApprovalState(TypedDict):
    task: str
    plan: str
    approved: bool
    result: str

# ── Nodes ──────────────────────────────────────────────────────────────────
def plan_task(state: ApprovalState) -> dict:
    plan = (
        f"Plan for '{state['task']}':\n"
        "  Step 1 — Validate inputs\n"
        "  Step 2 — Execute operation\n"
        "  Step 3 — Verify results"
    )
    print(f"\n📋 Generated plan:\n{plan}\n")
    approval = interrupt({
        "question": "Do you approve this plan?",
        "options": ["yes", "no", "modify"],
    })
    print(f"👤 Human responded: {approval}")
    return {"plan": plan, "approved": (approval == "yes")}

def execute_task(state: ApprovalState) -> dict:
    if not state["approved"]:
        return {"result": "❌ Task cancelled by user."}
    return {"result": f"✅ Executed: {state['task']}"}

# ── Graph ──────────────────────────────────────────────────────────────────
db = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(db)

builder = StateGraph(ApprovalState)
builder.add_node("plan", plan_task)
builder.add_node("execute", execute_task)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)
app = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "approval-live-01"}}

# ── Run to interrupt ───────────────────────────────────────────────────────
app.invoke(
    {"task": "Deploy database migration", "plan": "", "approved": False, "result": ""},
    config,
)

# ── Prompt the human ───────────────────────────────────────────────────────
human_answer = input("your decision").strip().lower()
# ── Resume the graph with the real human input ─────────────────────────────
final = app.invoke(Command(resume=human_answer), config)
print(f"\n🏁 Final result: {final['result']}")


📋 Generated plan:
Plan for 'Deploy database migration':
  Step 1 — Validate inputs
  Step 2 — Execute operation
  Step 3 — Verify results



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



📋 Generated plan:
Plan for 'Deploy database migration':
  Step 1 — Validate inputs
  Step 2 — Execute operation
  Step 3 — Verify results

👤 Human responded: no

🏁 Final result: ❌ Task cancelled by user.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

def step1(state: MessagesState) -> dict:
    return {"messages": [{"role": "assistant", "content": "Step 1 complete"}]}

def step2(state: MessagesState) -> dict:
    return {"messages": [{"role": "assistant", "content": "Step 2 complete"}]}

stream_builder = StateGraph(MessagesState)
stream_builder.add_node("step1", step1)
stream_builder.add_node("step2", step2)
stream_builder.add_edge(START, "step1")
stream_builder.add_edge("step1", "step2")
stream_builder.add_edge("step2", END)
stream_app = stream_builder.compile()

print("=== stream_mode='values' — full state after each node ===")
for chunk in stream_app.stream(
    {"messages": [HumanMessage("Go")]},
    stream_mode="values"
):
    print(f"State has {len(chunk['messages'])} messages")
# print("\n=== stream_mode='updates' — only changes from each node ===")
# for chunk in stream_app.stream(
#     {"messages": [HumanMessage("Go")]},
#     stream_mode="updates"
# ):
#     print(f"Node update: {chunk}")

=== stream_mode='values' — full state after each node ===
State has 1 messages
State has 2 messages
State has 3 messages


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

class ProcessState(TypedDict):
    items: list
    processed: list
def process_items(state: ProcessState) -> dict:
    writer = get_stream_writer()
    processed = []
    for i,item in enumerate(state["items"]):
        result = f"processed_{item}"
        writer({""})

=== stream_mode='custom' — custom events from nodes ===
Progress: alpha (1/3)
Progress: beta (2/3)
Progress: gamma (3/3)


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)

def chat(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    print(response)
    return {"messages": [response]}

chat_builder = StateGraph(MessagesState)
chat_builder.add_node("chat", chat)
chat_builder.add_edge(START, "chat")
chat_builder.add_edge("chat", END)
chat_app = chat_builder.compile()

print("=== stream_mode='messages' — LLM tokens as they arrive ===")
print("Response: ", end="", flush=True)
for chunk, metadata in chat_app.stream(
    {"messages": [HumanMessage("Count from 1 to 20, one number per word.")]},
    stream_mode="messages"
):
    if hasattr(chunk, "content") and chunk.content:
        print(chunk.content, end="", flush=True)
# print()  # newline after streaming

=== stream_mode='messages' — LLM tokens as they arrive ===
Response: 

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


One, two, three, four, five, six, seven, eight, nine, ten, eleven, twelve, thirteen, fourteen, fifteen, sixteen, seventeen, eighteen, nineteen, twenty.content='One, two, three, four, five, six, seven, eight, nine, ten, eleven, twelve, thirteen, fourteen, fifteen, sixteen, seventeen, eighteen, nineteen, twenty.' additional_kwargs={} response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9aeefbb7d1', 'service_tier': 'default', 'model_provider': 'openai'} id='lc_run--01a0630d-b336-77b3-bd98-2b7195163fe9' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 20, 'output_tokens': 40, 'total_tokens': 60, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [11]:
from typing_extensions import TypedDict
from typing import Annotated
from langchain_core.tools import tool
from langgraph.graph.message import add_messages

class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    findings: list
    approved_sources: list

@tool
def search_papers(query: str) -> str:
    """Search for academic papers (simulated)."""
    papers = [
        "Paper 1: Introduction to Quantum Computing",
        "Paper 2: Quantum Algorithms Overview",
        "Paper 3: Practical Quantum Applications"
    ]
    return "\n".join(papers)

@tool
def summarize_paper(title: str) -> str:
    """Summarize a paper (simulated, expensive operation)."""
    return f"Summary of {title}: [Detailed summary would go here]"

def research_node(state: ResearchState) -> dict:
    """Conduct research with human oversight."""
    # Search for papers
    results = search_papers.invoke({"query": state["topic"]})
    
    # Ask for clarification if needed
    clarification = interrupt(
        f"Found these papers about {state['topic']}. \n{results}\n\n"
        "Should I continue with detailed analysis? (yes/no)"
    )
    
    if clarification == "yes":
        # Proceed with analysis
        findings = ["Finding 1", "Finding 2", "Finding 3"]
        return {"findings": findings}
    
    return {"findings": []}

# Build research assistant
research_builder = StateGraph(ResearchState)
research_builder.add_node("research", research_node)
research_builder.add_edge(START, "research")
research_builder.add_edge("research", END)

research_db = sqlite3.connect(":memory:", check_same_thread=False)
research_memory = SqliteSaver(research_db)
research_assistant = research_builder.compile(checkpointer=research_memory)

print("✅ Research Assistant ready!")

✅ Research Assistant ready!


In [1]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import operator

load_dotenv()
print("✅ Ready for parallel execution!")

✅ Ready for parallel execution!


In [3]:
class BatchState(TypedDict):
    items: list
    results: Annotated[list,operator.add]
class ItemState(TypedDict):
    item: str  # Single item

def distribute_work(state:BatchState):
    return [Send("process_item",{"item":item}) for item in state['items']] 
def process_item(state:ItemState)-> dict:
    item=state["item"]
    result=f"Processed :{item.upper()}"
    return {"results":[result]}
parallel_graph = StateGraph(BatchState)
parallel_graph.add_node("process_item",process_item)
parallel_graph.add_conditional_edges(START,distribute_work,["process_item"])
parallel_graph.add_edge("process_item",END)

In [4]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# ── Two-level state design ────────────────────────────────
# OverallState: the global shared state of the whole graph
# WorkerState:  the private state injected per Send() call

class OverallState(TypedDict):
    documents:  list[str]           # input: list of docs to review
    reviews:   Annotated[list[str], operator.add]  # reducer: collects all results
    summary:   str

class WorkerState(TypedDict):
    document:  str   # ← each worker gets ONE document (not the full list)
    doc_index: int   # ← worker also knows which doc it is

# ── Nodes ─────────────────────────────────────────────────
def load_documents(state: OverallState):
    """Entry node: loads 4 documents into global state."""
    docs = [
        "Q4 Revenue Report: revenue up 18% YoY",
        "Risk Assessment: supply chain risks identified",
        "Product Roadmap: 3 major features planned for H1",
        "HR Update: headcount grew by 15 in Q4",
    ]
    print(f"📂 Loaded {len(docs)} documents.")
    return {"documents": docs}

def review_document(state: WorkerState):
    """
    Worker node — receives ONE document via Send().
    Note: 'state' here is WorkerState, NOT OverallState.
    Each parallel invocation of this node has a different 'document'.
    """
    doc   = state["document"]
    idx   = state["doc_index"]
    review = f"[Doc {idx+1}] ✅ Reviewed: {doc[:40]}..."
    print(f"  🔎 Worker {idx+1} reviewing doc...")
    return {"reviews": [review]}  # → appended to OverallState.reviews via reducer

def compile_summary(state: OverallState):
    """Fan-in node: receives all reviews merged by the reducer."""
    print(f"\n📝 All {len(state['reviews'])} reviews collected. Compiling summary...")
    summary = "SUMMARY: " + " | ".join(state["reviews"])
    return {"summary": summary}

# ── The KEY function: returns a list of Send() objects ────
def dispatch_reviewers(state: OverallState) -> list[Send]:
    """
    Called as a conditional edge function.
    Returns N Send objects → spawns N parallel review_document workers,
    each with its OWN custom state dict (different document + index).
    """
    return [
        Send("review_document", {"document": doc, "doc_index": i})
        for i, doc in enumerate(state["documents"])
    ]
    # Result: 4 parallel invocations of review_document, each with different doc

# ── Build Graph ───────────────────────────────────────────
builder = StateGraph(OverallState)
builder.add_node("load_documents",   load_documents)
builder.add_node("review_document",   review_document)
builder.add_node("compile_summary",   compile_summary)
builder.add_edge(START,               "load_documents")

# ← dispatch_reviewers returns [Send(...), Send(...), ...]
builder.add_conditional_edges("load_documents", dispatch_reviewers, ["review_document"])
builder.add_edge("review_document",   "compile_summary")
builder.add_edge("compile_summary",   END)
graph = builder.compile()

result = graph.invoke({"documents": [], "reviews": [], "summary": ""})
print(f"\n✅ Final summary:\n{result['summary']}")

📂 Loaded 4 documents.
  🔎 Worker 1 reviewing doc...
  🔎 Worker 2 reviewing doc...
  🔎 Worker 3 reviewing doc...
  🔎 Worker 4 reviewing doc...

📝 All 4 reviews collected. Compiling summary...

✅ Final summary:
SUMMARY: [Doc 1] ✅ Reviewed: Q4 Revenue Report: revenue up 18% YoY... | [Doc 2] ✅ Reviewed: Risk Assessment: supply chain risks iden... | [Doc 3] ✅ Reviewed: Product Roadmap: 3 major features planne... | [Doc 4] ✅ Reviewed: HR Update: headcount grew by 15 in Q4...


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [7]:
# ============================================================
# SEND API — EXAMPLE 1: DIFFERENT NODE PER ITEM
# Most people think Send only fans out to ONE node type.
# But each Send() call can target a DIFFERENT node.
# Here: each ticket routes to a specialist node based on its type.
# ============================================================

import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class OverallState(TypedDict):
    tickets:  list[dict]
    resolved: Annotated[list[str], operator.add]

class TicketState(TypedDict):
    ticket_id: str
    message:  str
    priority: str

# ── Three different specialist nodes ─────────────────────
def handle_billing(state: TicketState):
    print(f"  💳 BILLING  agent → ticket {state['ticket_id']}: '{state['message'][:30]}'")
    return {"resolved": [f"[BILLING]  {state['ticket_id']} resolved"]}

def handle_tech(state: TicketState):
    print(f"  🔧 TECH     agent → ticket {state['ticket_id']}: '{state['message'][:30]}'")
    return {"resolved": [f"[TECH]     {state['ticket_id']} resolved"]}

def handle_escalation(state: TicketState):
    print(f"  🚨 ESCALATE agent → ticket {state['ticket_id']}: '{state['message'][:30]}' [PRIORITY: {state['priority']}]")
    return {"resolved": [f"[ESCALATE] {state['ticket_id']} escalated to manager"]}

def load_tickets(state: OverallState):
    tickets = [
        {"ticket_id": "T001", "type": "billing",    "priority": "low",      "message": "I was charged twice this month"},
        {"ticket_id": "T002", "type": "tech",       "priority": "medium",   "message": "App crashes on startup"},
        {"ticket_id": "T003", "type": "billing",    "priority": "critical", "message": "Fraudulent charge on account"},
        {"ticket_id": "T004", "type": "tech",       "priority": "low",      "message": "Dark mode not working"},
        {"ticket_id": "T005", "type": "tech",       "priority": "critical", "message": "Data loss after update"},
    ]
    print(f"📥 Loaded {len(tickets)} support tickets.")
    return {"tickets": tickets}

# ── KEY: Each item dispatches to a DIFFERENT node ─────────
def dispatch_tickets(state: OverallState) -> list[Send]:
    sends = []
    for t in state["tickets"]:
        worker_state = {"ticket_id": t["ticket_id"],
                        "message":   t["message"],
                        "priority":  t["priority"]}

        # CRITICAL tickets → escalation node (regardless of type)
        # billing tickets  → billing node
        # tech tickets     → tech node
        if t["priority"] == "critical":
            sends.append(Send("handle_escalation", worker_state))  # ← different node!
        elif t["type"] == "billing": 
            sends.append(Send("handle_billing",    worker_state))  # ← different node!
        else:
            sends.append(Send("handle_tech",       worker_state))  # ← different node!
    print(f"\n🚦 Dispatching {len(sends)} tickets to different specialist nodes in parallel...")
    print(sends)

    return sends

def report(state: OverallState):
    print(f"\n📊 All {len(state['resolved'])} tickets resolved:")
    for r in sorted(state["resolved"]):
        print(f"   {r}")
    return {}
builder = StateGraph(OverallState)
builder.add_node("load_tickets",      load_tickets)
builder.add_node("handle_billing",    handle_billing)
builder.add_node("handle_tech",       handle_tech)
builder.add_node("handle_escalation", handle_escalation)
builder.add_node("report",             report)
builder.add_edge(START,                "load_tickets")
builder.add_conditional_edges(
    "load_tickets", dispatch_tickets,
    ["handle_billing", "handle_tech", "handle_escalation"]  # ← declare all possible targets
)
# All 3 specialist nodes converge at report
builder.add_edge("handle_billing",    "report")
builder.add_edge("handle_tech",       "report")
builder.add_edge("handle_escalation", "report")
builder.add_edge("report",             END)

graph = builder.compile()
graph.invoke({"tickets": [], "resolved": []})

📥 Loaded 5 support tickets.

🚦 Dispatching 5 tickets to different specialist nodes in parallel...
[Send(node='handle_billing', arg={'ticket_id': 'T001', 'message': 'I was charged twice this month', 'priority': 'low'}), Send(node='handle_tech', arg={'ticket_id': 'T002', 'message': 'App crashes on startup', 'priority': 'medium'}), Send(node='handle_escalation', arg={'ticket_id': 'T003', 'message': 'Fraudulent charge on account', 'priority': 'critical'}), Send(node='handle_tech', arg={'ticket_id': 'T004', 'message': 'Dark mode not working', 'priority': 'low'}), Send(node='handle_escalation', arg={'ticket_id': 'T005', 'message': 'Data loss after update', 'priority': 'critical'})]
  💳 BILLING  agent → ticket T001: 'I was charged twice this month'
  🔧 TECH     agent → ticket T002: 'App crashes on startup'
  🚨 ESCALATE agent → ticket T003: 'Fraudulent charge on account' [PRIORITY: critical]
  🚨 ESCALATE agent → ticket T005: 'Data loss after update' [PRIORITY: critical]
  🔧 TECH     agent → ti

{'tickets': [{'ticket_id': 'T001',
   'type': 'billing',
   'priority': 'low',
   'message': 'I was charged twice this month'},
  {'ticket_id': 'T002',
   'type': 'tech',
   'priority': 'medium',
   'message': 'App crashes on startup'},
  {'ticket_id': 'T003',
   'type': 'billing',
   'priority': 'critical',
   'message': 'Fraudulent charge on account'},
  {'ticket_id': 'T004',
   'type': 'tech',
   'priority': 'low',
   'message': 'Dark mode not working'},
  {'ticket_id': 'T005',
   'type': 'tech',
   'priority': 'critical',
   'message': 'Data loss after update'}],
 'resolved': ['[BILLING]  T001 resolved',
  '[TECH]     T002 resolved',
  '[ESCALATE] T003 escalated to manager',
  '[TECH]     T004 resolved',
  '[ESCALATE] T005 escalated to manager']}

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [5]:
# ============================================================
# SEND API — EXAMPLE 2: VARIABLE FAN-OUT COUNT
# The number of Send() calls is NOT fixed at build time.
# It is computed DYNAMICALLY from the current state at runtime.
# Example: Exam grader — spawns one grader per student (0 to N).
# Also shows: passing EXTRA metadata alongside the item.
# ============================================================

import operator, random
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class ExamState(TypedDict):
    exam_name:    str
    total_marks:  int
    submissions:  list[dict]           # dynamically filled
    grades:       Annotated[list[dict], operator.add]

class GraderState(TypedDict):
    student_id:   str
    answers:      list[int]           # raw scores per question
    exam_name:    str                 # ← extra metadata passed from OverallState
    total_marks:  int                 # ← extra metadata passed from OverallState

# ── Phase 1: Simulate collecting submissions ──────────────
def collect_submissions(state: ExamState):
    """
    Simulates receiving N submissions. N is random (3-7).
    The fan-out count will only be known HERE, at runtime.
    """
    random.seed(42)
    n_students   = random.randint(4, 7)
    submissions  = [
        {
            "student_id": f"STU-{i+1:03d}",
            "answers":    [random.randint(0, 10) for _ in range(5)]  # 5 questions
        }
        for i in range(n_students)
    ]
    print(f"📋 Collected {n_students} submissions for '{state['exam_name']}'")
    return {"submissions": submissions}

# ── Dispatch: VARIABLE number of graders ──────────────────
def dispatch_graders(state: ExamState) -> list[Send]:
    """
    Returns len(submissions) Send objects — one per student.
    Each worker also receives exam_name and total_marks as context.
    The exact number is ONLY known here at runtime!
    """
    n = len(state["submissions"])
    print(f"⚡ Spawning {n} grader workers dynamically...")
    return [
        Send("grade_submission", {
            "student_id":  sub["student_id"],
            "answers":     sub["answers"],
            # ↓ EXTRA METADATA passed alongside the item
            "exam_name":   state["exam_name"],
            "total_marks": state["total_marks"],
        })
        for sub in state["submissions"]
    ]

# ── Worker: grades one student's answers ─────────────────
def grade_submission(state: GraderState):
    raw       = sum(state["answers"])
    pct       = round(raw / state["total_marks"] * 100, 1)
    grade     = "A" if pct>=80 else "B" if pct>=65 else "C" if pct>=50 else "F"
    result    = {"student":state["student_id"], "score":raw, "pct":pct, "grade":grade}
    print(f"  📝 {state['student_id']}: {raw}/{state['total_marks']} = {pct}% → Grade {grade}")
    return {"grades": [result]}

# ── Reduce: compute class statistics ──────────────────────
def compute_stats(state: ExamState):
    grades  = state["grades"]
    avg     = round(sum(g["pct"] for g in grades) / len(grades), 1)
    top     = max(grades, key=lambda x: x["pct"])
    dist    = {}
    for g in grades:
        dist[g["grade"]] = dist.get(g["grade"], 0) + 1
    print(f"\n📊 {state['exam_name']} Results ({len(grades)} students):")
    print(f"   Class average : {avg}%")
    print(f"   Top student   : {top['student']} ({top['pct']}%)")
    print(f"   Grade dist    : {dist}")
    return {}

builder = StateGraph(ExamState)
builder.add_node("collect_submissions", collect_submissions)
builder.add_node("grade_submission",    grade_submission)
builder.add_node("compute_stats",       compute_stats)
builder.add_edge(START,                  "collect_submissions")
builder.add_conditional_edges("collect_submissions", dispatch_graders, ["grade_submission"])
builder.add_edge("grade_submission", "compute_stats")
builder.add_edge("compute_stats",    END)
graph = builder.compile()

graph.invoke({
    "exam_name":   "Advanced LLMs & RAG Midterm",
    "total_marks": 50,
    "submissions": [],
    "grades":      [],
})

📋 Collected 4 submissions for 'Advanced LLMs & RAG Midterm'
⚡ Spawning 4 grader workers dynamically...
  📝 STU-001: 12/50 = 24.0% → Grade F
  📝 STU-002: 29/50 = 58.0% → Grade C
  📝 STU-003: 10/50 = 20.0% → Grade F
  📝 STU-004: 28/50 = 56.0% → Grade C

📊 Advanced LLMs & RAG Midterm Results (4 students):
   Class average : 39.5%
   Top student   : STU-002 (58.0%)
   Grade dist    : {'F': 2, 'C': 2}


{'exam_name': 'Advanced LLMs & RAG Midterm',
 'total_marks': 50,
 'submissions': [{'student_id': 'STU-001', 'answers': [0, 4, 3, 3, 2]},
  {'student_id': 'STU-002', 'answers': [1, 10, 8, 1, 9]},
  {'student_id': 'STU-003', 'answers': [6, 0, 0, 1, 3]},
  {'student_id': 'STU-004', 'answers': [3, 8, 9, 0, 8]}],
 'grades': [{'student': 'STU-001', 'score': 12, 'pct': 24.0, 'grade': 'F'},
  {'student': 'STU-002', 'score': 29, 'pct': 58.0, 'grade': 'C'},
  {'student': 'STU-003', 'score': 10, 'pct': 20.0, 'grade': 'F'},
  {'student': 'STU-004', 'score': 28, 'pct': 56.0, 'grade': 'C'}]}

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [5]:
# ============================================================
# SEND API — EXAMPLE 3 FIXED: CHAINED SEND (TWO-LEVEL FAN-OUT)
# 
# WHAT WAS WRONG:
# audit_department was used as BOTH a node (must return dict)
# AND a routing function (returns [Send(...)]). That's invalid.
#
# THE FIX:
# Introduce an intermediate fan-in node (collect_depts) that
# waits for ALL level 1 workers, then a separate dispatch
# function for level 2. Clean separation of concerns.
#
# Correct flow:
# start_audit → [dispatch_departments] → audit_department ×3 (parallel)
#                                               ↓ (fan-in)
#                                         collect_depts
#                                               ↓ [dispatch_employees]
#                                         audit_employee ×9 (parallel)
#                                               ↓ (fan-in)
#                                         compile_report → END
# ============================================================

import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# ── States ────────────────────────────────────────────────
class AuditState(TypedDict):
    company:   str
    dept_data: Annotated[list[dict], operator.add]  # level 1 workers write here
    audit_log: Annotated[list[str],  operator.add]  # all workers append here

class DeptState(TypedDict):  # private state for level 1 workers
    company:   str
    dept:      str
    employees: list[str]

class EmpState(TypedDict):   # private state for level 2 workers
    company:   str
    dept:      str
    employee:  str

# ── Level 0: Entry node ───────────────────────────────────
def start_audit(state: AuditState):
    print(f"🏢 Starting HR audit for: {state['company']}")
    return {"audit_log": [f"Audit started for {state['company']}"]}

# ── Level 0 → Level 1 dispatch ────────────────────────────
def dispatch_departments(state: AuditState) -> list[Send]:
    """FIRST fan-out: one Send per department."""
    departments = {
        "Engineering": ["Alice", "Bob", "Charlie"],
        "Marketing":   ["Diana", "Eve"],
        "Finance":     ["Frank", "Grace", "Henry", "Ivy"],
    }
    print(f"\n  📂 Fan-out LEVEL 1: dispatching {len(departments)} dept auditors in parallel...")
    return [
        Send("audit_department", {
            "company":   state["company"],
            "dept":      dept,
            "employees": emps,
        })
        for dept, emps in departments.items()
    ]

# ── Level 1 worker: RETURNS A DICT (not Send objects!) ────
def audit_department(state: DeptState):
    """
    FIX: This node now correctly returns a dict.
    It stores employee data into 'dept_data' via the reducer
    so the level 2 dispatcher can read them later.
    """
    print(f"  🗂  [{state['dept']}] auditing dept — {len(state['employees'])} employees found")
    return {
        "dept_data": [{            # ← written to global AuditState via operator.add
            "company":   state["company"],
            "dept":      state["dept"],
            "employees": state["employees"],
        }],
        "audit_log": [f"Dept audited: {state['dept']} ({len(state['employees'])} employees)"],
    }

# ── Intermediate fan-in: waits for ALL level 1 workers ────
def collect_depts(state: AuditState):
    """
    FIX: This new intermediate node runs AFTER all audit_department
    workers complete and their results are merged into AuditState.
    It's a passthrough — just confirms collection before level 2 dispatch.
    """
    total = sum(len(d["employees"]) for d in state["dept_data"])
    print(f"\n  ✅ All {len(state['dept_data'])} depts collected. Total employees: {total}")
    return {}

# ── Level 1 → Level 2 dispatch ────────────────────────────
def dispatch_employees(state: AuditState) -> list[Send]:
    """
    SECOND fan-out: reads dept_data that was collected from all
    level 1 workers and spawns one Send per employee.
    """
    sends = [
        Send("audit_employee", {
            "company":  d["company"],
            "dept":     d["dept"],
            "employee": emp,
        })
        for d in state["dept_data"]
        for emp in d["employees"]
    ]
    print(f"  📂 Fan-out LEVEL 2: dispatching {len(sends)} employee auditors in parallel...")
    return sends

# ── Level 2 worker: leaf node, returns dict ───────────────
def audit_employee(state: EmpState):
    entry = f"✓ {state['company']} | {state['dept']:12s} | {state['employee']}"
    print(f"    👤 {entry}")
    return {"audit_log": [entry]}

# ── Final fan-in: compile report ──────────────────────────
def compile_report(state: AuditState):
    emp_logs = [l for l in state["audit_log"] if l.startswith("✓")]
    print(f"\n📋 AUDIT COMPLETE — {len(emp_logs)} employees audited:")
    for log in sorted(emp_logs):
        print(f"   {log}")
    return {}

# ── Build Graph ───────────────────────────────────────────
builder = StateGraph(AuditState)
builder.add_node("start_audit",       start_audit)
builder.add_node("audit_department",  audit_department)   # level 1: returns dict ✅
builder.add_node("collect_depts",     collect_depts)      # intermediate fan-in node ✅
builder.add_node("audit_employee",    audit_employee)     # level 2: returns dict ✅
builder.add_node("compile_report",    compile_report)

builder.add_edge(START,                "start_audit")
# Level 1 fan-out
builder.add_conditional_edges("start_audit",   dispatch_departments, ["audit_department"])
# Fan-in: all level 1 workers → collect_depts
builder.add_edge("audit_department",           "collect_depts")
# Level 2 fan-out
builder.add_conditional_edges("collect_depts", dispatch_employees,   ["audit_employee"])
# Fan-in: all level 2 workers → compile_report
builder.add_edge("audit_employee",             "compile_report")
builder.add_edge("compile_report",             END)

graph = builder.compile()
graph.invoke({"company": "KrishAI Technologies", "dept_data": [], "audit_log": []})

🏢 Starting HR audit for: KrishAI Technologies

  📂 Fan-out LEVEL 1: dispatching 3 dept auditors in parallel...
  🗂  [Engineering] auditing dept — 3 employees found
  🗂  [Marketing] auditing dept — 2 employees found
  🗂  [Finance] auditing dept — 4 employees found

  ✅ All 3 depts collected. Total employees: 9
  📂 Fan-out LEVEL 2: dispatching 9 employee auditors in parallel...
    👤 ✓ KrishAI Technologies | Engineering  | Alice
    👤 ✓ KrishAI Technologies | Engineering  | Bob
    👤 ✓ KrishAI Technologies | Engineering  | Charlie
    👤 ✓ KrishAI Technologies | Marketing    | Eve
    👤 ✓ KrishAI Technologies | Finance      | Frank
    👤 ✓ KrishAI Technologies | Marketing    | Diana
    👤 ✓ KrishAI Technologies | Finance      | Grace
    👤 ✓ KrishAI Technologies | Finance      | Henry
    👤 ✓ KrishAI Technologies | Finance      | Ivy

📋 AUDIT COMPLETE — 9 employees audited:
   ✓ KrishAI Technologies | Engineering  | Alice
   ✓ KrishAI Technologies | Engineering  | Bob
   ✓ KrishAI Technol

{'company': 'KrishAI Technologies',
 'dept_data': [{'company': 'KrishAI Technologies',
   'dept': 'Engineering',
   'employees': ['Alice', 'Bob', 'Charlie']},
  {'company': 'KrishAI Technologies',
   'dept': 'Marketing',
   'employees': ['Diana', 'Eve']},
  {'company': 'KrishAI Technologies',
   'dept': 'Finance',
   'employees': ['Frank', 'Grace', 'Henry', 'Ivy']}],
 'audit_log': ['Audit started for KrishAI Technologies',
  'Dept audited: Engineering (3 employees)',
  'Dept audited: Marketing (2 employees)',
  'Dept audited: Finance (4 employees)',
  '✓ KrishAI Technologies | Engineering  | Alice',
  '✓ KrishAI Technologies | Engineering  | Bob',
  '✓ KrishAI Technologies | Engineering  | Charlie',
  '✓ KrishAI Technologies | Marketing    | Diana',
  '✓ KrishAI Technologies | Marketing    | Eve',
  '✓ KrishAI Technologies | Finance      | Frank',
  '✓ KrishAI Technologies | Finance      | Grace',
  '✓ KrishAI Technologies | Finance      | Henry',
  '✓ KrishAI Technologies | Finance   

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [6]:
# ============================================================
# SEND API — EXAMPLE 4: WORKERS ROUTE TO DIFFERENT DOWNSTREAM NODES
# Combines Send API (for fan-out) with Command(goto=...) inside
# the worker node to dynamically pick the NEXT node per item.
# Pipeline: detect content type → parallel workers → each worker
# routes to its own post-processing node based on result quality.
# Example: RAG pipeline — retrieve N chunks, each gets scored,
# high-score chunks go to "use_chunk", low-score to "discard_chunk".
# ============================================================

import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send, Command

class RAGState(TypedDict):
    query:    str
    chunks:   list[dict]
    used:     Annotated[list[str], operator.add]   # kept chunks
    discarded:Annotated[list[str], operator.add]   # dropped chunks
    answer:   str

class ChunkState(TypedDict):
    query:    str
    chunk_id: str
    text:     str
    score:    float   # relevance score pre-computed

# ── Entry: simulate retrieval ─────────────────────────────
def retrieve_chunks(state: RAGState):
    chunks = [
        {"chunk_id":"C1","text":"LangGraph supports stateful multi-agent workflows","score":0.91},
        {"chunk_id":"C2","text":"The weather in Paris is 22 degrees today",       "score":0.12},
        {"chunk_id":"C3","text":"Send API enables dynamic parallel node dispatch",   "score":0.88},
        {"chunk_id":"C4","text":"Recipe for pasta carbonara requires eggs and cheese","score":0.08},
        {"chunk_id":"C5","text":"Checkpointers enable persistence across graph runs", "score":0.76},
        {"chunk_id":"C6","text":"Stock price of AAPL closed at $189 yesterday",      "score":0.31},
    ]
    print(f"📥 Retrieved {len(chunks)} chunks for query: '{state['query']}'")
    return {"chunks": chunks}

# ── Dispatch: one worker per chunk ────────────────────────
def dispatch_scorers(state: RAGState) -> list[Send]:
    print(f"\n⚡ Dispatching {len(state['chunks'])} chunk scorers in parallel...")
    return [
        Send("score_chunk", {
            "query":    state["query"],
            "chunk_id": c["chunk_id"],
            "text":     c["text"],
            "score":    c["score"],
        })
        for c in state["chunks"]
    ]

# ── Worker: scores chunk AND routes to different next nodes ─
def score_chunk(state: ChunkState) -> Command[Literal["use_chunk", "discard_chunk"]]:
    """
    Worker receives one chunk via Send().
    Based on score, routes to a DIFFERENT downstream node via Command.
    → score >= 0.6  : use_chunk   (high quality)
    → score <  0.6  : discard_chunk (low quality)
    This is the KEY insight: Send + Command together = dynamic fan-out
    with per-item routing to different nodes.
    """
    threshold = 0.6
    if state["score"] >= threshold:
        print(f"  ✅ {state['chunk_id']} score={state['score']:.2f} → use_chunk")
        return Command(
            update={"used": [f"{state['chunk_id']}: {state['text'][:45]}..."]},
            goto="use_chunk",
        )
    else:
        print(f"  ❌ {state['chunk_id']} score={state['score']:.2f} → discard_chunk")
        return Command(
            update={"discarded": [f"{state['chunk_id']}: score too low ({state['score']:.2f})"]},
            goto="discard_chunk",
        )

def use_chunk(state: RAGState):
    print(f"  📎 use_chunk: adding to context window")
    return {}

def discard_chunk(state: RAGState):
    print(f"  🗑  discard_chunk: dropping from context")
    return {}

def generate_answer(state: RAGState):
    print(f"\n🤖 Generating answer from {len(state['used'])} high-quality chunks")
    print(f"   Used    : {[c.split(':')[0] for c in state['used']]}")
    print(f"   Discarded: {[c.split(':')[0] for c in state['discarded']]}")
    answer = f"Answer using {len(state['used'])} relevant chunks: [LLM response here]"
    return {"answer": answer}

builder = StateGraph(RAGState)
builder.add_node("retrieve_chunks", retrieve_chunks)
builder.add_node("score_chunk",     score_chunk)
builder.add_node("use_chunk",       use_chunk)
builder.add_node("discard_chunk",   discard_chunk)
builder.add_node("generate_answer", generate_answer)
builder.add_edge(START,              "retrieve_chunks")
builder.add_conditional_edges("retrieve_chunks", dispatch_scorers, ["score_chunk"])
# score_chunk uses Command(goto=...) internally, so no edge needed from it
builder.add_edge("use_chunk",       "generate_answer")
builder.add_edge("discard_chunk",   "generate_answer")
builder.add_edge("generate_answer", END)
graph = builder.compile()

graph.invoke({
    "query":     "How does LangGraph handle state persistence?",
    "chunks":    [],
    "used":      [],
    "discarded": [],
    "answer":    "",
})

📥 Retrieved 6 chunks for query: 'How does LangGraph handle state persistence?'

⚡ Dispatching 6 chunk scorers in parallel...
  ✅ C1 score=0.91 → use_chunk
  ✅ C3 score=0.88 → use_chunk
  ❌ C2 score=0.12 → discard_chunk
  ❌ C4 score=0.08 → discard_chunk
  ❌ C6 score=0.31 → discard_chunk
  ✅ C5 score=0.76 → use_chunk
  🗑  discard_chunk: dropping from context
  📎 use_chunk: adding to context window

🤖 Generating answer from 3 high-quality chunks
   Used    : ['C1', 'C3', 'C5']
   Discarded: ['C2', 'C4', 'C6']


{'query': 'How does LangGraph handle state persistence?',
 'chunks': [{'chunk_id': 'C1',
   'text': 'LangGraph supports stateful multi-agent workflows',
   'score': 0.91},
  {'chunk_id': 'C2',
   'text': 'The weather in Paris is 22 degrees today',
   'score': 0.12},
  {'chunk_id': 'C3',
   'text': 'Send API enables dynamic parallel node dispatch',
   'score': 0.88},
  {'chunk_id': 'C4',
   'text': 'Recipe for pasta carbonara requires eggs and cheese',
   'score': 0.08},
  {'chunk_id': 'C5',
   'text': 'Checkpointers enable persistence across graph runs',
   'score': 0.76},
  {'chunk_id': 'C6',
   'text': 'Stock price of AAPL closed at $189 yesterday',
   'score': 0.31}],
 'used': ['C1: LangGraph supports stateful multi-agent workf...',
  'C3: Send API enables dynamic parallel node dispat...',
  'C5: Checkpointers enable persistence across graph...'],
 'discarded': ['C2: score too low (0.12)',
  'C4: score too low (0.08)',
  'C6: score too low (0.31)'],
 'answer': 'Answer using 3 releva

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [7]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# ── Subgraph A: Preprocessing Pipeline ────────────────────
# Has its own isolated state with private keys
class PreprocState(TypedDict):
    text:      str   # shared with parent
    char_count:int   # private — only inside subgraph A

def clean_text(state: PreprocState):
    cleaned = state["text"].strip().lower()
    print(f"  [SubA] clean_text: '{cleaned[:30]}...'")
    return {"text": cleaned}

def count_chars(state: PreprocState):
    n = len(state["text"])
    print(f"  [SubA] count_chars: {n} chars")
    return {"char_count": n}

preproc_builder = StateGraph(PreprocState)
preproc_builder.add_node("clean_text",  clean_text)
preproc_builder.add_node("count_chars", count_chars)
preproc_builder.add_edge(START,          "clean_text")
preproc_builder.add_edge("clean_text",  "count_chars")
preproc_builder.add_edge("count_chars", END)
preproc_subgraph = preproc_builder.compile()   # ← compiled = ready to be a node

# ── Subgraph B: Analysis Pipeline ─────────────────────────
class AnalysisState(TypedDict):
    text:      str    # shared with parent
    keywords: list[str]  # private — only inside subgraph B

def extract_keywords(state: AnalysisState):
    words    = [w for w in state["text"].split() if len(w) > 5]
    print(f"  [SubB] extract_keywords: {words[:4]}")
    return {"keywords": words}

def score_sentiment(state: AnalysisState):
    pos_words = ["great", "good", "excellent", "amazing", "fantastic"]
    score     = sum(1 for w in state["keywords"] if w in pos_words)
    label     = "positive" if score > 0 else "neutral"
    # Only 'text' is shared back to parent — 'keywords' stays private
    print(f"  [SubB] score_sentiment: {label}")
    return {"text": f"[{label.upper()}] {state['text']}"}

analysis_builder = StateGraph(AnalysisState)
analysis_builder.add_node("extract_keywords", extract_keywords)
analysis_builder.add_node("score_sentiment",  score_sentiment)
analysis_builder.add_edge(START,               "extract_keywords")
analysis_builder.add_edge("extract_keywords", "score_sentiment")
analysis_builder.add_edge("score_sentiment",  END)
analysis_subgraph = analysis_builder.compile()

# ── Parent Graph ───────────────────────────────────────────
# Uses both subgraphs as plain nodes via add_node()
class ParentState(TypedDict):
    text:   str   # shared key — subgraphs read & write this
    result: str

def format_result(state: ParentState):
    print(f"  [Parent] Formatting result.")
    return {"result": f"DONE: {state['text']}"}

parent = StateGraph(ParentState)
parent.add_node("preprocess", preproc_subgraph)    # ← subgraph as node!
parent.add_node("analyse",     analysis_subgraph)   # ← subgraph as node!
parent.add_node("format",      format_result)
parent.add_edge(START,         "preprocess")
parent.add_edge("preprocess",  "analyse")
parent.add_edge("analyse",     "format")
parent.add_edge("format",      END)
graph = parent.compile()

print("🚀 Running parent graph with 2 subgraphs...\n")
out = graph.invoke({"text": "  LangGraph is Great for building Agentic AI systems!  ", "result": ""})
print(f"\n✅ {out['result']}")

🚀 Running parent graph with 2 subgraphs...

  [SubA] clean_text: 'langgraph is great for buildin...'
  [SubA] count_chars: 51 chars
  [SubB] extract_keywords: ['langgraph', 'building', 'agentic', 'systems!']
  [SubB] score_sentiment: neutral
  [Parent] Formatting result.

✅ DONE: [NEUTRAL] langgraph is great for building agentic ai systems!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [8]:
# ============================================================
# MAP-REDUCE IN LANGGRAPH
# MAP phase  : Send API fans out N parallel workers (one per item).
# REDUCE phase: Annotated[list, operator.add] reducer collects all results.
# The reduce node only runs AFTER all N map workers complete.
# Classic pattern: generate → score → pick best.
# Example: Generate 4 essay outlines in parallel, pick the best one.
# ============================================================

import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# ── State Design ─────────────────────────────────────────
class OverallState(TypedDict):
    topic:       str
    approaches: list[str]       # the N items to fan-out over
    # Annotated + operator.add is the REDUCE mechanism
    # All worker outputs append to this list automatically
    outlines:   Annotated[list[dict], operator.add]
    best_outline:str

class WorkerState(TypedDict):
    topic:    str
    approach: str    # each worker gets a different approach

# ── PHASE 1: Generate N approaches (the setup) ────────────
def generate_approaches(state: OverallState):
    """Generates the list of items to map over."""
    approaches = [
        "chronological narrative",
        "problem-solution framework",
        "compare and contrast",
        "case study based",
    ]
    print(f"📋 Generated {len(approaches)} essay approaches to evaluate in parallel.")
    return {"approaches": approaches}

# ── MAP: Dispatch one worker per approach via Send() ──────
def dispatch_workers(state: OverallState) -> list[Send]:
    """MAP: Fan-out — one Send per approach = N parallel workers."""
    return [
        Send("write_outline", {"topic": state["topic"], "approach": a})
        for a in state["approaches"]
    ]

# ── Worker node: runs N times in parallel ─────────────────
def write_outline(state: WorkerState):
    """Each worker creates one outline with a score."""
    approach = state["approach"]
    topic    = state["topic"]

    # Simulate scoring (would be an LLM evaluator in production)
    scores = {
        "chronological narrative":    72,
        "problem-solution framework":  91,
        "compare and contrast":        68,
        "case study based":            85,
    }
    score = scores.get(approach, 50)
    outline = {
        "approach": approach,
        "score":    score,
        "preview":  f"Essay on '{topic}' using {approach}",
    }
    print(f"  ✍️  Worker [{approach}] → score: {score}")
    return {"outlines": [outline]}  # ← appended to OverallState.outlines

# ── REDUCE: Runs after ALL workers finish ─────────────────
def pick_best_outline(state: OverallState):
    """
    REDUCE phase: receives ALL outlines merged by operator.add.
    Picks the highest-scoring one.
    """
    print(f"\n🔀 REDUCE: received {len(state['outlines'])} outlines.")
    best = max(state["outlines"], key=lambda x: x["score"])
    print(f"🏆 Best approach: '{best['approach']}' (score: {best['score']})")
    return {"best_outline": best["preview"]}

# ── Build Graph ───────────────────────────────────────────
builder = StateGraph(OverallState)
builder.add_node("generate_approaches", generate_approaches)
builder.add_node("write_outline",       write_outline)       # MAP worker
builder.add_node("pick_best_outline",   pick_best_outline)   # REDUCE
builder.add_edge(START,                   "generate_approaches")

# Conditional edge returns [Send(...),...] = the MAP phase
builder.add_conditional_edges("generate_approaches", dispatch_workers, ["write_outline"])

# All workers → reduce node (fan-in)
builder.add_edge("write_outline",       "pick_best_outline")
builder.add_edge("pick_best_outline",   END)
graph = builder.compile()

print("🚀 Running map-reduce essay outliner...\n")
result = graph.invoke({
    "topic":        "The Impact of Agentic AI on Software Engineering",
    "approaches":  [],
    "outlines":    [],
    "best_outline":"",
})
print(f"\n✅ Best: {result['best_outline']}")

🚀 Running map-reduce essay outliner...

📋 Generated 4 essay approaches to evaluate in parallel.
  ✍️  Worker [chronological narrative] → score: 72
  ✍️  Worker [problem-solution framework] → score: 91
  ✍️  Worker [compare and contrast] → score: 68
  ✍️  Worker [case study based] → score: 85

🔀 REDUCE: received 4 outlines.
🏆 Best approach: 'problem-solution framework' (score: 91)

✅ Best: Essay on 'The Impact of Agentic AI on Software Engineering' using problem-solution framework


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [9]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from dotenv import load_dotenv
import sqlite3
import hashlib
import time

load_dotenv()
print("✅ Ready for production patterns!")

✅ Ready for production patterns!


In [12]:
# Simple in-memory cache
CACHE = {}

class CachedState(TypedDict):
    query: str
    result: str
    cache_hit: bool

def cached_search(state: CachedState) -> dict:
    """Search with caching to avoid redundant API calls."""
    query = state["query"]
    
    # Create cache key
    cache_key = hashlib.md5(query.encode()).hexdigest()
    print(CACHE)
    # Check cache
    if cache_key in CACHE:
        print(f"✅ Cache HIT for: {query}")
        return {
            "result": CACHE[cache_key],
            "cache_hit": True
        }
    
    # Cache miss - perform expensive operation
    print(f"❌ Cache MISS for: {query}")
    print("   Performing expensive search...")
    time.sleep(1)  # Simulate expensive operation
    
    result = f"Search results for: {query}"
    
    # Store in cache
    CACHE[cache_key] = result
    
    return {
        "result": result,
        "cache_hit": False
    }

# Build cached graph
cache_builder = StateGraph(CachedState)
cache_builder.add_node("search", cached_search)
cache_builder.add_edge(START, "search")
cache_builder.add_edge("search", END)
cached_app = cache_builder.compile()

# Test caching
print("\nTest 1: First call (cache miss)")
result1 = cached_app.invoke({"query": "LangGraph tutorial", "result": "", "cache_hit": False})

print("\nTest 2: Same query (cache hit)")
result2 = cached_app.invoke({"query": "LangGraph tutorial", "result": "", "cache_hit": False})

print(f"\n📊 Cache effectiveness:")
print(f"   First call: {result1['cache_hit']}")
print(f"   Second call: {result2['cache_hit']}")
print(f"   First call: {result1['result']}")
print(f"   Second call: {result2['result']}")


Test 1: First call (cache miss)
{}
❌ Cache MISS for: LangGraph tutorial
   Performing expensive search...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



Test 2: Same query (cache hit)
{'333a1e6fe39e6d1d209957fd214fedbe': 'Search results for: LangGraph tutorial'}
✅ Cache HIT for: LangGraph tutorial

📊 Cache effectiveness:
   First call: False
   Second call: True
   First call: Search results for: LangGraph tutorial
   Second call: Search results for: LangGraph tutorial


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
class GuardedState(TypedDict):
    messages: Annotated[list,add_messages]
    filtered: bool
BLOCKED_WORDS =["hack"]
def pre_model_hook(state: GuardedState)->dict:
    last_message = state["messages"][-1].content
    if any(word in last_message.lower() for word in BLOCKED_WORDS):
        return{
            "filtered":True,
            "messages":[SystemMessage(content="I cannot help")]
        }
def call_model_with_hooks(state:GuardedState)->dict:
    pre_result=pre_model_hook(state)
    if  pre_result.get("filtered"):
        return pre_result   

In [ ]:
try:
    from langgraph.callbacks import GraphLifecycleHandler
    class CustomObservabilityHandler(GraphLifecycleHandler):
          

Note: GraphLifecycleHandler requires langgraph>=1.1.7
The standard approach is to use LangSmith for observability.


In [ ]:
from typing_extensions import TypedDict
from typing import Annotated
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END

class ResilientState(TypedDict):
    task: str
    result: str
    retry_count: int
    max_retries: int
    error: str

def resilient_task(state: ResilientState) -> dict:
    """Task with error handling and retries."""
    try:
        print(f"🔄 Attempt {state['retry_count'] + 1}/{state['max_retries']}")
        
        # Simulate operation that might fail
        import random
        if random.random() < 0.9:  # 50% failure rate
            raise Exception("Simulated error")
        
        # Success
        result = f"Completed: {state['task']}"
        print("✅ Success!")
        return {"result": result, "error": ""}
        
    except Exception as e:
        print(f"❌ Error: {e}")
        
        # Increment retry count
        new_count = state["retry_count"] + 1
        
        if new_count >= state["max_retries"]:
            # Max retries exceeded
            error_msg = f"Failed after {new_count} attempts: {e}"
            return {
                "result": "",
                "error": error_msg,
                "retry_count": new_count
            }
        
        # Retry
        return {"retry_count": new_count, "error": str(e)}

from typing import Literal

def should_retry(state:ResilientState) -> Literal["task","__end__"]:
    if state["result"] or state["retry_count"] >=state["max_retries"]:
        return

# Build resilient graph
resilient_builder = StateGraph(ResilientState)
resilient_builder.add_node("task", resilient_task)
resilient_builder.add_edge(START, "task")
resilient_builder.add_conditional_edges("task", should_retry)
resilient_app = resilient_builder.compile()

# Test resilient execution
result = resilient_app.invoke({
    "task": "Important operation",
    "result": "",
    "retry_count": 0,
    "max_retries": 5,
    "error": ""
})

print("\n📊 Final state:")
print(f"   Result: {result['result']}")
print(f"   Attempts: {result['retry_count']}")
print(f"   Error: {result['error']}")

🔄 Attempt 1/5
❌ Error: Simulated error
🔄 Attempt 2/5
❌ Error: Simulated error
🔄 Attempt 3/5
❌ Error: Simulated error
🔄 Attempt 4/5
❌ Error: Simulated error
🔄 Attempt 5/5
❌ Error: Simulated error

📊 Final state:
   Result: 
   Attempts: 5
   Error: Failed after 5 attempts: Simulated error


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [7]:
# Create a chatbot with history tracking
class HistoryState(TypedDict):
    messages: Annotated[list, add_messages]
    step_count: int

def history_chat(state: HistoryState) -> dict:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
    response = llm.invoke(state["messages"])
    return {
        "messages": [response],
        "step_count": state.get("step_count", 0) + 1
    }

history_builder = StateGraph(HistoryState)
history_builder.add_node("chat", history_chat)
history_builder.add_edge(START, "chat")
history_builder.add_edge("chat", END)

# Compile with persistent checkpointer for history
history_db = sqlite3.connect(":memory:", check_same_thread=False)
history_memory = SqliteSaver(history_db)
history_app = history_builder.compile(checkpointer=history_memory)

# Have a multi-turn conversation
config = {"configurable": {"thread_id": "history-demo"}}

print("Having a conversation...")
history_app.invoke(
    {"messages": [HumanMessage(content="Hi, I'm learning LangGraph")], "step_count": 0},
    config
)

history_app.invoke(
    {"messages": [HumanMessage(content="What are the key concepts?")]},
    config
)

history_app.invoke(
    {"messages": [HumanMessage(content="Tell me about checkpointing")]},
    config
)

# Get state history
print("\n📜 State History:")
print("=" * 60)

states = list(history_app.get_state_history(config))
print(f"Total checkpoints: {len(states)}")

for i, state in enumerate(reversed(states[:5])):
    print(f"\nCheckpoint {i+1}:")
    print(f"   Step: {state.values.get('step_count', 0)}")
    print(f"   Messages: {len(state.values.get('messages', []))}")

NameError: name 'sqlite3' is not defined

In [ ]:


class SupportState(TypedDict):
    message: str
    category: str
    priority: str
    response: str

def classify_ticket(state:SupportState) -> Command:
    message = state["message"].lower()
    if any(word in message for word in ["billing", "charge", "refund", "payment"]):
        category = "billing"
        priority = "high"
    return Command(
        update={"category":category,"priority":priority},
        goto=f"hadndle_{category}"
    )

In [14]:
from typing import Literal
from langchain_openai import ChatOpenAI
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

class EnterpriseState(TypedDict):
    messages: Annotated[list, add_messages]
    ticket_id: str
    severity: str  # low, medium, high, critical
    category: str  # technical, billing, general
    kb_cached: bool
    escalated: bool
    resolution: str

# Knowledge base cache
KB_CACHE = {}

@tool
def search_knowledge_base(query: str) -> str:
    """Search KB with caching."""
    cache_key = hashlib.md5(query.encode()).hexdigest()
    
    if cache_key in KB_CACHE:
        print("   💾 KB cache hit")
        return KB_CACHE[cache_key]
    
    print("   🔍 KB cache miss - searching...")
    # Simulated KB
    kb_result = f"KB article for: {query}"
    KB_CACHE[cache_key] = kb_result
    return kb_result

# Build enterprise support system
support_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)
support_llm_with_tools = support_llm.bind_tools([search_knowledge_base])

def classify_ticket(state: EnterpriseState) -> dict:
    """Classify ticket severity and category."""
    last_msg = state["messages"][-1].content
    
    # Simple classification (in production, use LLM)
    severity = "high" if any(word in last_msg.lower() for word in ["urgent", "down", "critical"]) else "medium"
    category = "technical" if any(word in last_msg.lower() for word in ["error", "bug", "crash"]) else "general"
    
    print(f"🏷️  Classified: {severity} severity, {category} category")
    return {"severity": severity, "category": category}

def support_agent(state: EnterpriseState) -> dict:
    """Main support agent with guardrails."""
    # Pre-hook: validate input
    print("🔒 Validating input...")
    
    # Call LLM with tools
    response = support_llm_with_tools.invoke(state["messages"])
    
    # Post-hook: check output
    print("🔒 Validating output...")
    
    return {"messages": [response]}

def route_support(state: EnterpriseState) -> Literal["tools", "escalate", "__end__"]:
    """Route based on severity and tool calls."""
    last_message = state["messages"][-1]
    print(last_message)
    
    # Check for escalation
    if state["severity"] == "critical":
        return "escalate"
    
    # Check for tool calls
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    
    return "__end__"

def call_support_tools(state: EnterpriseState) -> dict:
    """Execute support tools."""
    from langchain_core.messages import ToolMessage
    
    last_message = state["messages"][-1]
    tool_messages = []
    
#    last_message=AIMessage(
#     tool_calls=[
#         {
#             "name": "search_knowledge_base",
#             "args": {
#                 "query": "How can I unlock a locked account?"
#             },
#             "id": "call_123"
#         }
#     ]
# )
    
    for tool_call in last_message.tool_calls:
        result = search_knowledge_base.invoke(tool_call["args"])
        tool_messages.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )
    
    return {"messages": tool_messages, "kb_cached": True}

def escalate(state: EnterpriseState) -> dict:
    """Escalate to human support."""
    print("🚨 Escalating to human support...")
    return {
        "escalated": True,
        "resolution": "Escalated to senior support team"
    }

# Build enterprise graph
enterprise_builder = StateGraph(EnterpriseState)
enterprise_builder.add_node("classify", classify_ticket)
enterprise_builder.add_node("agent", support_agent)
enterprise_builder.add_node("tools", call_support_tools)
enterprise_builder.add_node("escalate", escalate)

enterprise_builder.add_edge(START, "classify")
enterprise_builder.add_edge("classify", "agent")
enterprise_builder.add_conditional_edges("agent", route_support)
enterprise_builder.add_edge("tools", "agent")
enterprise_builder.add_edge("escalate", END)

# Compile with production checkpointer
enterprise_db = sqlite3.connect("enterprise_support.db", check_same_thread=False)
enterprise_memory = SqliteSaver(enterprise_db)
enterprise_system = enterprise_builder.compile(checkpointer=enterprise_memory)

print("✅ Enterprise Customer Support System ready!")
print("\n🎯 Features enabled:")
print("   ✅ Multi-agent architecture")
print("   ✅ Persistent memory (SqliteSaver)")
print("   ✅ Knowledge base caching")
print("   ✅ Input/output validation")
print("   ✅ Automatic escalation")
print("   ✅ Tool integration")

✅ Enterprise Customer Support System ready!

🎯 Features enabled:
   ✅ Multi-agent architecture
   ✅ Persistent memory (SqliteSaver)
   ✅ Knowledge base caching
   ✅ Input/output validation
   ✅ Automatic escalation
   ✅ Tool integration


In [15]:
# Test the enterprise system
from langchain_core.messages import HumanMessage, SystemMessage

ticket_config = {"configurable": {"thread_id": "ticket-enterprise-001"}}

result = enterprise_system.invoke({
    "messages": [HumanMessage(content="I'm having trouble logging into my account. Getting error 401.")],
    "ticket_id": "ENT-001",
    "severity": "",
    "category": "",
    "kb_cached": False,
    "escalated": False,
    "resolution": ""
}, ticket_config)

print("\n" + "=" * 70)
print("🎫 Ticket ENT-001 - Support Response")
print("=" * 70)
print(f"Severity: {result['severity']}")
print(f"Category: {result['category']}")
print(f"Escalated: {result['escalated']}")
print(f"\nResponse: {result['messages'][-1].content}")
print("=" * 70)

🏷️  Classified: medium severity, technical category
🔒 Validating input...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


🔒 Validating output...
content='Error 401 typically indicates an "Unauthorized" error, which means the server did not recognize your credentials or you did not provide any. Here are a few steps you can take to resolve this issue:\n\n1. **Check Credentials**: Ensure that your username and password are entered correctly. Be aware of any case sensitivity or special characters that need to be included.\n\n2. **Reset Password**: If you\'re unsure about your password, try resetting it. Most services offer a "Forgot Password" link on their login page.\n\n3. **Clear Cache and Cookies**: Sometimes, clearing your browser\'s cache and cookies can resolve login issues.\n\n4. **Enable Cookies**: Make sure that cookies are enabled in your browser settings, as some websites require cookies for logging in.\n\n5. **Update Browser**: Ensure that your browser is up-to-date, as older versions might not be compatible with the website\'s security requirements.\n\n6. **Try a Different Browser or Device**: So

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

import langgraph


try:
    import langgraph_supervisor
    print("langgraph-supervisor: installed")
    SUPERVISOR_AVAILABLE = True
except ImportError:
    print("WARNING: langgraph-supervisor not installed. Run: uv pip install langgraph-supervisor>=0.1.0")
    SUPERVISOR_AVAILABLE = False

try:
    import langgraph_swarm
    print("langgraph-swarm: installed")
    SWARM_AVAILABLE = True
except ImportError:
    print("WARNING: langgraph-swarm not installed. Run: uv pip install langgraph-swarm>=0.1.0")
    SWARM_AVAILABLE = False

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not set. Add it to your .env file.")

print("Setup complete")

langgraph-supervisor: installed
langgraph-swarm: installed
Setup complete
